# Lab 12 — Trotterización y Evolución Temporal Guiada

La trotterización permite simular la evolución temporal de un sistema cuántico e^{-iHt}
cuando H = H₁ + H₂ y [H₁, H₂] ≠ 0.

$$e^{-i(H_1+H_2)t} \approx (e^{-iH_1 \delta t} e^{-iH_2 \delta t})^n + O(t^2/n)$$

**Aplicación**: simulación de la dinámica de spins, modelos de Ising, propagación de información cuántica.

## 1. Hamiltoniano de Heisenberg y evolución exacta

$$H = X_0 X_1 + Y_0 Y_1 + Z_0 Z_1$$

La evolución exacta se calcula exponenciando la matriz:
$$|\psi(t)\rangle = e^{-iHt}|\psi(0)\rangle$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp, Operator
from scipy.linalg import expm

# Hamiltoniano Heisenberg 2 spins
H = SparsePauliOp.from_list([('XX', 1.0), ('YY', 1.0), ('ZZ', 1.0)])
H_mat = H.to_matrix()

# Estado inicial |01⟩
psi0 = np.array([0, 1, 0, 0], dtype=complex)  # |01⟩

def evolve_exact(psi0: np.ndarray, H: np.ndarray, t: float) -> np.ndarray:
    U = expm(-1j * H * t)
    return U @ psi0

# Probabilidad de |01⟩ vs |10⟩ con el tiempo (intercambio de excitación)
times = np.linspace(0, 2 * np.pi, 200)
p01 = [abs(evolve_exact(psi0, H_mat, t)[1])**2 for t in times]
p10 = [abs(evolve_exact(psi0, H_mat, t)[2])**2 for t in times]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(times / np.pi, p01, 'b-', label='P(|01⟩)', linewidth=2)
ax.plot(times / np.pi, p10, 'r-', label='P(|10⟩)', linewidth=2)
ax.set_xlabel('t / π', fontsize=12)
ax.set_ylabel('Probabilidad', fontsize=12)
ax.set_title('Evolución exacta: intercambio de excitación en Heisenberg', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Trotterización de primer orden

Descomponemos H = H_XX + H_YY + H_ZZ y aplicamos cada término por separado:

$$e^{-iHt} \approx \left(e^{-iH_{XX}\delta t} e^{-iH_{YY}\delta t} e^{-iH_{ZZ}\delta t}\right)^n$$

El error Trotter es O(t²/n): aumentar n reduce el error.

In [ ]:
def trotter_step(delta_t: float) -> QuantumCircuit:
    """Un paso Trotter para H = XX + YY + ZZ."""
    qc = QuantumCircuit(2)
    # e^{-i*XX*dt}: CNOT + Rx(2dt) + CNOT
    qc.cx(0, 1)
    qc.rx(2 * delta_t, 0)
    qc.cx(0, 1)
    # e^{-i*YY*dt}: con puertas de base
    qc.sdg(0); qc.sdg(1)  # S†: |0⟩→|0⟩, |1⟩→-i|1⟩
    qc.cx(0, 1)
    qc.rx(2 * delta_t, 0)
    qc.cx(0, 1)
    qc.s(0); qc.s(1)
    # e^{-i*ZZ*dt}: CNOT + Rz(2dt) + CNOT
    qc.cx(0, 1)
    qc.rz(2 * delta_t, 1)
    qc.cx(0, 1)
    return qc

def trotter_evolve(psi0: np.ndarray, t: float, n_steps: int) -> np.ndarray:
    """Evolución trotterizada en n_steps pasos."""
    dt = t / n_steps
    step_mat = Operator(trotter_step(dt)).data
    psi = psi0.copy()
    for _ in range(n_steps):
        psi = step_mat @ psi
    return psi

# Comparar exacta vs Trotter con n=1,5,20
t_eval = np.pi / 2  # tiempo fijo
psi_exact = evolve_exact(psi0, H_mat, t_eval)
print(f'Estado exacto en t=π/2: {np.abs(psi_exact)**2.round(4)}')
for n in [1, 5, 10, 20, 50]:
    psi_trotter = trotter_evolve(psi0, t_eval, n)
    fidelity = abs(np.dot(psi_exact.conj(), psi_trotter))**2
    print(f'  n={n:2d}: F = {fidelity:.6f}')

## 3. Error Trotter vs número de pasos

El error de fidelidad escala como O(1/n²) para Trotter de primer orden (ya que la fidelidad es 1 − O(t²/n²) para tiempos cortos).

In [ ]:
t_fixed = np.pi
n_values = [1, 2, 4, 8, 16, 32, 64, 128]
psi_exact_t = evolve_exact(psi0, H_mat, t_fixed)
errors = []

for n in n_values:
    psi_t = trotter_evolve(psi0, t_fixed, n)
    fidelity = abs(np.dot(psi_exact_t.conj(), psi_t))**2
    errors.append(1 - fidelity)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(n_values, errors, 'bo-', markersize=8, linewidth=2, label='Error Trotter')
# Referencia O(1/n²)
ref = errors[0] * (n_values[0] / np.array(n_values))**2
ax.loglog(n_values, ref, 'r--', alpha=0.6, label='O(1/n²)')
ax.set_xlabel('Número de pasos n', fontsize=12)
ax.set_ylabel('Error de fidelidad 1-F', fontsize=12)
ax.set_title(f'Convergencia Trotter (t={t_fixed/np.pi:.1f}π)', fontsize=13)
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Trotterización de segundo orden (Suzuki)

La fórmula de Suzuki de segundo orden reduce el error a O(t³/n²):

$$S_2(\delta t) = e^{-iH_1\delta t/2} e^{-iH_2\delta t} e^{-iH_1\delta t/2}$$

Esto dobla el coste por paso pero cuadra el exponente de convergencia.

In [ ]:
def suzuki_step(delta_t: float) -> QuantumCircuit:
    """Paso Suzuki de 2° orden para H = XX + YY + ZZ."""
    qc = QuantumCircuit(2)

    def add_xx(qc, t):
        qc.cx(0, 1); qc.rx(2*t, 0); qc.cx(0, 1)

    def add_yy(qc, t):
        qc.sdg(0); qc.sdg(1)
        qc.cx(0, 1); qc.rx(2*t, 0); qc.cx(0, 1)
        qc.s(0); qc.s(1)

    def add_zz(qc, t):
        qc.cx(0, 1); qc.rz(2*t, 1); qc.cx(0, 1)

    # S2: H1/2, H2/2, H3, H2/2, H1/2 (simetría)
    add_xx(qc, delta_t/2)
    add_yy(qc, delta_t/2)
    add_zz(qc, delta_t)
    add_yy(qc, delta_t/2)
    add_xx(qc, delta_t/2)
    return qc

def suzuki_evolve(psi0, t, n_steps):
    dt = t / n_steps
    step_mat = Operator(suzuki_step(dt)).data
    psi = psi0.copy()
    for _ in range(n_steps):
        psi = step_mat @ psi
    return psi

print('Comparación Trotter 1° vs Suzuki 2° orden:')
print(f'n  | Trotter error | Suzuki error')
print('-' * 40)
for n in [1, 2, 4, 8, 16]:
    err_t = 1 - abs(np.dot(psi_exact_t.conj(), trotter_evolve(psi0, t_fixed, n)))**2
    err_s = 1 - abs(np.dot(psi_exact_t.conj(), suzuki_evolve(psi0, t_fixed, n)))**2
    print(f'{n:2d} | {err_t:.2e}      | {err_s:.2e}')

## 5. Resumen

| Método | Error por paso | Puertas por paso | Escalado |
|--------|---------------|-----------------|----------|
| Trotter 1° | O(δt²) | k | O(kn) |
| Suzuki 2° | O(δt³) | 2k-1 | O(kn) con mejor constante |
| Suzuki p° | O(δt^{p+1}) | O(5^{p/2}) | Exponencial en p |

**Regla práctica**: usar Suzuki de 2° orden es casi siempre mejor que Trotter de 1° con el mismo número de puertas.

In [ ]:
# Verificar escala de convergencia
suzuki_errors = []
for n in n_values:
    psi_s = suzuki_evolve(psi0, t_fixed, n)
    suzuki_errors.append(1 - abs(np.dot(psi_exact_t.conj(), psi_s))**2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(n_values, errors, 'bo-', label='Trotter 1°: O(1/n²)')
ax.loglog(n_values, suzuki_errors, 'rs-', label='Suzuki 2°: O(1/n⁴)')
ref2 = [max(e, 1e-15) for e in suzuki_errors]
ax.loglog(n_values, errors[0]*(n_values[0]/np.array(n_values))**2, 'b--', alpha=0.4)
ax.loglog(n_values, ref2[0]*(n_values[0]/np.array(n_values))**4, 'r--', alpha=0.4)
ax.set_xlabel('Pasos n', fontsize=12)
ax.set_ylabel('Error 1-F', fontsize=12)
ax.set_title('Convergencia Trotter vs Suzuki', fontsize=13)
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()